# 数据预处理、增强与数据泄漏

## 学习目标

掌握训练/验证/测试数据的边界，理解标准化统计量的来源，并实现一个只在训练阶段启用的增强流程。

## 概念模型

预处理要保持输入契约一致；增强只能改变训练样本，不能污染验证和测试。任何从数据估计的统计量都只能在训练集上计算。

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
raw = torch.randn(30, 1, 8, 8) * 4 + 10
labels = (raw.mean(dim=(1, 2, 3)) > 10).long()
train_raw, val_raw = raw[:20], raw[20:]
train_y, val_y = labels[:20], labels[20:]
mean = train_raw.mean()
std = train_raw.std().clamp_min(1e-6)
print('train statistics:', round(mean.item(), 3), round(std.item(), 3))

### 实验 1：训练集统计量和独立 transform

验证集只使用训练集的 mean/std，不重新 fit；否则验证信息会泄漏到训练流程。

In [ ]:
def normalize(x):
    return (x - mean) / std
train_x = normalize(train_raw)
val_x = normalize(val_raw)
assert abs(train_x.mean().item()) < 0.2
print('normalized shapes:', train_x.shape, val_x.shape)

class ImageDataset(Dataset):
    def __init__(self, images, targets, training=False):
        self.images, self.targets, self.training = images, targets, training
    def __len__(self): return len(self.targets)
    def __getitem__(self, index):
        image = self.images[index].clone()
        if self.training and torch.rand(()) < 0.5:
            image = image.flip(-1)
        return image, self.targets[index]

train_set = ImageDataset(train_x, train_y, training=True)
val_set = ImageDataset(val_x, val_y, training=False)
print('datasets:', len(train_set), len(val_set))

### 实验 2：batch 边界和类别分布

DataLoader 负责组 batch，不负责改变标签语义。训练集可以 shuffle，验证和测试通常不需要。

In [ ]:
train_loader = DataLoader(train_set, batch_size=6, shuffle=True)
val_loader = DataLoader(val_set, batch_size=6, shuffle=False)
batch_x, batch_y = next(iter(train_loader))
print('batch:', batch_x.shape, batch_y.shape, 'labels:', torch.bincount(batch_y, minlength=2).tolist())
assert batch_x.ndim == 4 and batch_y.ndim == 1

counts = torch.bincount(train_y, minlength=2).float()
class_weights = counts.sum() / counts.clamp_min(1)
class_weights = class_weights / class_weights.mean()
print('class weights:', class_weights.tolist())

## 检查点

解释为什么不能用训练集和验证集混合后的 mean/std；说明为什么验证集不能启用随机增强；列出类别不平衡时至少两种处理方法。

## 试一试

让验证 Dataset 错误地启用增强，重复读取同一个样本并比较结果；再用 `CrossEntropyLoss(weight=class_weights)` 训练一个小模型。

## 常见错误与调试

训练/验证 transform 共用可变状态、标准化统计量来自全数据、增强改变标签语义、标签 dtype 不符合损失函数要求、时间序列数据被随机打乱造成泄漏。